In [60]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import cv2
import os

In [66]:
"""
test_sample : 분석할 이미지가 있는 path
/crops : 이미지를 yolo를 통해 분석 후 바운딩 박스대로 crop한 후 해당 경로에 저장
merge_img_1.jpg : 모든 분석이 끝난 이미지 위에 바운딩 박스가 올라간 이미지 -> 이 이미지를 분석 후에 보여주면 좋을 듯.

"""

'\ntest_sample : 분석할 이미지가 있는 path\n/crops : 이미지를 yolo를 통해 분석 후 바운딩 박스대로 crop한 후 해당 경로에 저장\nmerge_img_1.jpg : 모든 분석이 끝난 이미지 위에 바운딩 박스가 올라간 이미지 -> 이 이미지를 분석 후에 보여주면 좋을 듯.\n\n'

In [61]:
model_path = "yolo_model_weight/best.pt"
model = torch.hub.load('ultralytics/yolov5', 'custom', path=model_path)

test_sample = "sample_data"

Using cache found in /Users/seyeong/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2024-8-24 Python-3.8.18 torch-2.4.1 CPU

Fusing layers... 
YOLOv5s summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 


In [62]:
pill_name = [
    'ApurtranTablet150mg(Irbesartan)',
    'BalsanTablet80mg(Valsartan)',
    'BalthrepTablet160mg(Valsartan)',
    'DiosartanTablet160mg(Valsartan)',
    'EscitalTablet5mg(EscitalopramOxalate)',
    'ExbanTablet80mg(Valsartan)',
    'FeratracTablet2.5mg(Letrozole)',
    'GasbetTablet5mg(MosaprideCitrateHydrate)',
    'GasdialTablet50mg(DimethiconeMagnesium)',
    'GasprenTablet(MosaprideCitrateDihydrate)',
    'GasridTablet5mg(MosaprideCitrateHydrate)',
    'LipinonTablet80mg(AtorvastatinCalciumTrihydrate)',
    'NumentaminSustainedReleaseCapsule8mg(GalantamineBromide)',
    'RosorodTablet10mg(RosuvastatinCalcium)',
    'SarvaltanTablet160mg(Valsartan)',
    'SurosinDTablet(TamsulosinHydrochloride)',
    'ValsartanTablet(Valsartan)',
    'ValsartelTablet160mg(Valsartan)',
    'ValsartelTablet80mg(Valsartan)',
    'ZolpidemSustainedReleaseTablet(ZolpidemTartrate)'
]
detected_pills = []
bounding_boxs = []
output_image_path = test_sample + '/merge_img_1.jpg'

In [63]:
def draw_bboxes(image_path, bounding_boxs_, output_path=None):
    image = cv2.imread(image_path)
    h, w, _ = image.shape
    
    for i, line in enumerate(bounding_boxs_):
        x1, y1, x2, y2 = line
        
        color = (0, 255, 0)  # 초록색 바운딩 박스
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(image, f'{detected_pills[i]}', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    if output_path:
        cv2.imwrite(output_path, image)
    else:
        cv2.imshow('Image with Bounding Boxes', image)
        cv2.waitKey(0)
        cv2.destroyAllWindows()

In [64]:
def test_backend(img_dir):
    from tensorflow.keras.models import load_model

    model = load_model('Pill_image_mobile_net.h5')

    image = Image.open(img_dir)
    image = image.resize((100, 100))
    image = np.array(image)
    image = image/255.
    
    plt.imshow(image)
    plt.show()
    
    image = np.reshape(image, (1, 100, 100, 3))
    
    prediction = model.predict(image)
    prediction.shape
    pred_class = np.argmax(prediction, axis=-1)

    print(pill_name[int(pred_class)])
    return pill_name[int(pred_class)][:]
    

In [65]:
def detect_and_crop_objects(image_path):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = model(img_rgb)

    bbox_list = results.xyxy[0].cpu().numpy()
    cropped_images = []

    for idx, bbox in enumerate(bbox_list):
        xmin, ymin, xmax, ymax, confidence, cls = map(int, bbox)
        
        cropped_img = img[ymin:ymax, xmin:xmax]
        cropped_images.append(cropped_img)  

        crop_output_path = os.path.join(test_sample, "crops")
        os.makedirs(crop_output_path, exist_ok=True)
        crop_file_path = os.path.join(crop_output_path, f"{os.path.basename(image_path).split('.')[0]}_crop_{idx}.jpg")
        cv2.imwrite(crop_file_path, cropped_img)

        bounding_boxs.append([xmin, ymin, xmax, ymax])

        pills = test_backend(crop_file_path)
        detected_pills.append(pills)

for image_name in sorted(os.listdir(test_sample)):
    if image_name.endswith(('.jpg', '.png', '.jpeg')):
        image_path = os.path.join(test_sample, image_name)
        detect_and_crop_objects(image_path)
        draw_bboxes(image_path, bounding_boxs, output_image_path)
        break


/Users/seyeong/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:869: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


1/1 [==============================] - 0s 300ms/step
BalsanTablet80mg(Valsartan)
1/1 [==============================] - 0s 287ms/step
DiosartanTablet160mg(Valsartan)
1/1 [==============================] - 0s 304ms/step
ValsartelTablet80mg(Valsartan)
1/1 [==============================] - 0s 279ms/step
GasprenTablet(MosaprideCitrateDihydrate)
1/1 [==============================] - 0s 299ms/step
ExbanTablet80mg(Valsartan)
1/1 [==============================] - 0s 296ms/step
RosorodTablet10mg(RosuvastatinCalcium)
1/1 [==============================] - 0s 282ms/step
GasprenTablet(MosaprideCitrateDihydrate)
1/1 [==============================] - 0s 284ms/step
DiosartanTablet160mg(Valsartan)


In [45]:
detected_pills

['BalsanTablet80mg(Valsartan)',
 'DiosartanTablet160mg(Valsartan)',
 'ValsartelTablet80mg(Valsartan)',
 'GasprenTablet(MosaprideCitrateDihydrate)',
 'ExbanTablet80mg(Valsartan)',
 'RosorodTablet10mg(RosuvastatinCalcium)',
 'GasprenTablet(MosaprideCitrateDihydrate)',
 'DiosartanTablet160mg(Valsartan)']

In [46]:
bounding_boxs

[[195, 69, 289, 159],
 [370, 232, 459, 307],
 [488, 361, 585, 458],
 [60, 197, 142, 292],
 [280, 521, 380, 620],
 [144, 542, 229, 626],
 [440, 47, 513, 140],
 [147, 451, 237, 510]]